# Sales 2025 SKU Real vs Edit Visualization

Notebook ini fokus ke satu tujuan: melihat dampak penyamaan SKU terhadap trend `CF` di `Sales 2025`.

Rule alignment:
- BIG 1625 / 1.625L -> BIG 1L terbaru
- BIG 3100 / 3.1L -> BIG 3L terbaru
- BIG 400ml dengan flavour sama -> SKU 400ml terbaru
- BIG Nipis 350ml -> deskripsi terbaru, SKU tetap sama
- VOLT 200ml -> SKU/deskripsi 24-pack terbaru
- Jika SKU 12-pack diarahkan ke 24-pack, `CF Edit = CF Real / 2`

In [ ]:
from pathlib import Path
import re
import zipfile

import numpy as np
import pandas as pd

try:
    import plotly.express as px
except ModuleNotFoundError:
    px = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

## 1. Source File

Notebook ini hanya memakai file dari Google Drive. Ubah path di bawah kalau lokasi file di Drive berbeda.

In [ ]:
DRIVE_SALES_FILE = Path("/content/drive/MyDrive/Data Sales/Indonesia Sales Dashboard 2026.xlsm")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Run this notebook in Colab, or mount Google Drive manually before running.")

if not DRIVE_SALES_FILE.exists():
    raise FileNotFoundError(f"File not found: {DRIVE_SALES_FILE}")

with zipfile.ZipFile(DRIVE_SALES_FILE) as zf:
    if "[Content_Types].xml" not in zf.namelist():
        raise ValueError(f"Not a readable Excel workbook: {DRIVE_SALES_FILE}")

SALES_FILE = DRIVE_SALES_FILE
print("Using:", SALES_FILE)

## 2. Load Data

Chart hanya memakai `Sales 2025`. `Sales 2026` dipakai sebagai acuan SKU terbaru.

In [ ]:
sales25 = pd.read_excel(SALES_FILE, sheet_name="Sales 2025", engine="openpyxl")
sales26 = pd.read_excel(SALES_FILE, sheet_name="Sales 2026", engine="openpyxl")

print("Sales 2025:", sales25.shape)
print("Sales 2026 reference:", sales26.shape)
display(sales25.head(3))

## 3. Clean Real SKU

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    return re.sub(r"\s+", " ", value)


def clean_code(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    return value[:-2] if value.endswith(".0") else value


def group_key(brand, flavor, fmt):
    brand = clean_text(brand)
    flavor = clean_text(flavor)
    if pd.isna(fmt):
        return ""
    fmt = float(fmt)

    if brand == "BIG" and "NIPIS" in flavor and np.isclose(fmt, 0.35):
        return "BIG_NIPIS_350"
    if brand == "BIG" and np.isclose(fmt, 0.4):
        return "BIG_400"
    if brand == "BIG" and (np.isclose(fmt, 1.625) or np.isclose(fmt, 1.0)):
        return "BIG_1L"
    if brand == "BIG" and (np.isclose(fmt, 3.1) or np.isclose(fmt, 3.0)):
        return "BIG_3L"
    if brand == "VOLT" and np.isclose(fmt, 0.2):
        return "VOLT_200_24"
    return ""


actual = pd.DataFrame({
    "Date": pd.to_datetime(sales25["Date"], errors="coerce"),
    "Channel Group": sales25["Channel Group"],
    "Branch": sales25["Branch"],
    "Channel": sales25["Channel"],
    "Cust Code": sales25["Cust Code"].map(clean_code),
    "Customer Name": sales25["Customer Name"],
    "Brand": sales25["Brand"].map(clean_text),
    "Flavor": sales25["Flavor"].map(clean_text),
    "Format": pd.to_numeric(sales25["Format"], errors="coerce"),
    "Box Content": sales25["Box Content"].map(clean_code),
    "CF": pd.to_numeric(sales25["CF"], errors="coerce").fillna(0),
    "Item Code Real": sales25["Item Code (Real)"].map(clean_code),
    "Short Item Description Real": sales25["Short Item Description (Edit).1"].map(clean_text),
})

actual = actual.dropna(subset=["Date"]).copy()
actual["Month"] = actual["Date"].dt.to_period("M").dt.to_timestamp()
actual["Group Key"] = [group_key(b, f, s) for b, f, s in zip(actual["Brand"], actual["Flavor"], actual["Format"])]
actual["SKU Real"] = actual["Item Code Real"] + " | " + actual["Short Item Description Real"]

display(actual.head(10))

## 4. Build Latest SKU Reference

Reference diambil dari `Sales 2026`, lalu dipilih row terbaru per Brand + Flavor + Group Key. Untuk VOLT, reference dibatasi ke `Box Content = 24`.

In [ ]:
ref = pd.DataFrame({
    "Reference Date": pd.to_datetime(sales26["fecha_liquidacion"], errors="coerce"),
    "Brand": sales26["desc_marca"].map(clean_text),
    "Flavor": sales26["desc_sabor"].map(clean_text),
    "Format": pd.to_numeric(sales26["desc_formato"], errors="coerce"),
    "Box Content": sales26["cant_contenido"].map(clean_code),
    "CF": pd.to_numeric(sales26["CF"], errors="coerce").fillna(0),
    "Item Code Edit": sales26["cod_articulo"].map(clean_code),
    "Short Item Description Edit": sales26["desc_articulo_corto"].map(clean_text),
})

ref = ref.dropna(subset=["Reference Date"]).copy()
ref["Group Key"] = [group_key(b, f, s) for b, f, s in zip(ref["Brand"], ref["Flavor"], ref["Format"])]
ref = ref[ref["Group Key"] != ""].copy()
ref = ref[(ref["Group Key"] != "VOLT_200_24") | (ref["Box Content"] == "24")]

sku_reference = (
    ref.sort_values(["Brand", "Flavor", "Group Key", "Reference Date", "CF"], ascending=[True, True, True, False, False])
    .drop_duplicates(["Brand", "Flavor", "Group Key"])
    [[
        "Brand", "Flavor", "Group Key", "Reference Date",
        "Item Code Edit", "Short Item Description Edit", "Format", "Box Content",
    ]]
    .rename(columns={"Format": "Reference Format", "Box Content": "Reference Box Content"})
)

display(sku_reference.sort_values(["Brand", "Group Key", "Flavor"]))

## 5. Apply SKU Edit

In [ ]:
aligned = actual.merge(sku_reference, on=["Brand", "Flavor", "Group Key"], how="left")
has_reference = aligned["Group Key"].ne("") & aligned["Item Code Edit"].notna()

aligned["Item Code Edit"] = np.where(has_reference, aligned["Item Code Edit"], aligned["Item Code Real"])
aligned["Short Item Description Edit"] = np.where(
    has_reference,
    aligned["Short Item Description Edit"],
    aligned["Short Item Description Real"],
)
aligned["SKU Edit"] = aligned["Item Code Edit"].map(clean_code) + " | " + aligned["Short Item Description Edit"].map(clean_text)
aligned["Real != Edit"] = aligned["SKU Real"] != aligned["SKU Edit"]
aligned["CF Real"] = aligned["CF"]
pack_12_to_24 = aligned["Box Content"].eq("12") & aligned["Reference Box Content"].eq("24") & aligned["Real != Edit"]
aligned["CF Edit"] = np.where(pack_12_to_24, aligned["CF Real"] / 2, aligned["CF Real"])
aligned["CF Adjustment"] = np.where(pack_12_to_24, "12-pack to 24-pack: CF / 2", "No CF adjustment")

sales_2025_visual = aligned[[
    "Date", "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()
sales_2025_visual.insert(0, "Source Sheet", "Sales 2025")
sales_2025_visual.insert(1, "Year", sales_2025_visual["Date"].dt.year)
sales_2025_visual.insert(2, "Month No", sales_2025_visual["Date"].dt.month)
sales_2025_visual["Month"] = sales_2025_visual["Date"].dt.to_period("M").dt.to_timestamp()

print("Rows:", len(sales_2025_visual))
print("Rows changed Real -> Edit:", int(sales_2025_visual["Real != Edit"].sum()))
print("Rows with CF / 2 adjustment:", int((sales_2025_visual["CF Adjustment"] == "12-pack to 24-pack: CF / 2").sum()))
display(sales_2025_visual.head(10))

## 6. Mapping Summary

In [ ]:
mapping_summary = (
    sales_2025_visual[sales_2025_visual["Real != Edit"]]
    .groupby(["Group Key", "SKU Real", "SKU Edit"], as_index=False)
    .agg(
        rows=("CF Real", "size"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
        cf_adjustment=("CF Adjustment", lambda s: ", ".join(sorted(set(s)))),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .sort_values(["Group Key", "SKU Real"])
)

display(mapping_summary)

## 7. Sales 2026 Alignment

In [ ]:
actual26 = pd.DataFrame({
    "Date": pd.to_datetime(sales26["fecha_liquidacion"], errors="coerce"),
    "Channel Group": sales26["DescCanalLocal (grupo)"],
    "Branch": sales26["desc_sucursal"],
    "Channel": sales26["DescCanalLocal"],
    "Cust Code": sales26["cod_cliente"].map(clean_code),
    "Customer Name": sales26["nomb_cliente"],
    "Brand": sales26["desc_marca"].map(clean_text),
    "Flavor": sales26["desc_sabor"].map(clean_text),
    "Format": pd.to_numeric(sales26["desc_formato"], errors="coerce"),
    "Box Content": sales26["cant_contenido"].map(clean_code),
    "CF": pd.to_numeric(sales26["CF"], errors="coerce").fillna(0),
    "Item Code Real": sales26["cod_articulo"].map(clean_code),
    "Short Item Description Real": sales26["desc_articulo_corto"].map(clean_text),
})

actual26 = actual26.dropna(subset=["Date"]).copy()
actual26["Month"] = actual26["Date"].dt.to_period("M").dt.to_timestamp()
actual26["Group Key"] = [group_key(b, f, s) for b, f, s in zip(actual26["Brand"], actual26["Flavor"], actual26["Format"])]
actual26["SKU Real"] = actual26["Item Code Real"] + " | " + actual26["Short Item Description Real"]

aligned26 = actual26.merge(sku_reference, on=["Brand", "Flavor", "Group Key"], how="left")
has_reference26 = aligned26["Group Key"].ne("") & aligned26["Item Code Edit"].notna()

aligned26["Item Code Edit"] = np.where(has_reference26, aligned26["Item Code Edit"], aligned26["Item Code Real"])
aligned26["Short Item Description Edit"] = np.where(
    has_reference26,
    aligned26["Short Item Description Edit"],
    aligned26["Short Item Description Real"],
)
aligned26["SKU Edit"] = aligned26["Item Code Edit"].map(clean_code) + " | " + aligned26["Short Item Description Edit"].map(clean_text)
aligned26["Real != Edit"] = aligned26["SKU Real"] != aligned26["SKU Edit"]
aligned26["CF Real"] = aligned26["CF"]
pack_12_to_24_26 = aligned26["Box Content"].eq("12") & aligned26["Reference Box Content"].eq("24") & aligned26["Real != Edit"]
aligned26["CF Edit"] = np.where(pack_12_to_24_26, aligned26["CF Real"] / 2, aligned26["CF Real"])
aligned26["CF Adjustment"] = np.where(pack_12_to_24_26, "12-pack to 24-pack: CF / 2", "No CF adjustment")

sales_2026_visual = aligned26[[
    "Date", "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()
sales_2026_visual.insert(0, "Source Sheet", "Sales 2026")
sales_2026_visual.insert(1, "Year", sales_2026_visual["Date"].dt.year)
sales_2026_visual.insert(2, "Month No", sales_2026_visual["Date"].dt.month)
sales_2026_visual["Month"] = sales_2026_visual["Date"].dt.to_period("M").dt.to_timestamp()

mapping_summary_2026 = (
    sales_2026_visual[sales_2026_visual["Real != Edit"]]
    .groupby(["Group Key", "SKU Real", "SKU Edit"], as_index=False)
    .agg(
        rows=("CF Real", "size"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
        cf_adjustment=("CF Adjustment", lambda s: ", ".join(sorted(set(s)))),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .sort_values(["Group Key", "SKU Real"])
)

print("Sales 2026 rows:", len(sales_2026_visual))
print("Sales 2026 rows changed Real -> Edit:", int(sales_2026_visual["Real != Edit"].sum()))
print("Sales 2026 rows with CF / 2 adjustment:", int((sales_2026_visual["CF Adjustment"] == "12-pack to 24-pack: CF / 2").sum()))
display(mapping_summary_2026)

## 8. Append Sales 2025 + Sales 2026

In [ ]:
append_columns = [
    "Source Sheet", "Year", "Month No", "Date",
    "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]

sales_aligned_append = pd.concat(
    [
        sales_2025_visual[append_columns],
        sales_2026_visual[append_columns],
    ],
    ignore_index=True,
)
sales_aligned_append["Month"] = sales_aligned_append["Date"].dt.to_period("M").dt.to_timestamp()

print("Appended rows:", len(sales_aligned_append))
print("Sales 2025 rows:", len(sales_2025_visual))
print("Sales 2026 rows:", len(sales_2026_visual))
print("Total rows changed Real -> Edit:", int(sales_aligned_append["Real != Edit"].sum()))
display(sales_aligned_append.head(10))

## 9. Appended Monthly CF by SKU - Edit

In [ ]:
append_edit_monthly = sales_aligned_append.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_append_edit = append_edit_monthly.groupby("SKU Edit")["CF"].sum().nlargest(20).index
append_edit_top = append_edit_monthly[append_edit_monthly["SKU Edit"].isin(top_append_edit)]

if px:
    fig = px.line(append_edit_top, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(append_edit_top.head(50))

## 10. Sales 2026 Monthly CF by SKU - Edit

In [ ]:
edit_monthly_2026 = sales_2026_visual.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_edit_2026 = edit_monthly_2026.groupby("SKU Edit")["CF"].sum().nlargest(20).index
edit_top_2026 = edit_monthly_2026[edit_monthly_2026["SKU Edit"].isin(top_edit_2026)]

if px:
    fig = px.line(edit_top_2026, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(edit_top_2026.head(50))

## 11. Distinct SKU Count: Real vs Edit

In [ ]:
distinct_monthly = (
    sales_aligned_append.groupby("Month")
    .agg(
        real_sku_count=("SKU Real", "nunique"),
        edit_sku_count=("SKU Edit", "nunique"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
    )
    .reset_index()
)

display(distinct_monthly)

if px:
    fig = px.line(
        distinct_monthly,
        x="Month",
        y=["real_sku_count", "edit_sku_count"],
        markers=True,
        title="Distinct SKU Count per Month: Real vs Edit - Sales 2025-2026",
    )
    fig.show()

## 12. Monthly CF by SKU - Real

In [ ]:
real_monthly = sales_aligned_append.groupby(["Month", "SKU Real"], as_index=False).agg(CF=("CF Real", "sum"))
top_real = real_monthly.groupby("SKU Real")["CF"].sum().nlargest(20).index
real_top = real_monthly[real_monthly["SKU Real"].isin(top_real)]

if px:
    fig = px.line(real_top, x="Month", y="CF", color="SKU Real", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Real Top 20")
    fig.show()
else:
    display(real_top.head(50))

## 13. Monthly CF by SKU - Edit

In [ ]:
edit_monthly = sales_aligned_append.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_edit = edit_monthly.groupby("SKU Edit")["CF"].sum().nlargest(20).index
edit_top = edit_monthly[edit_monthly["SKU Edit"].isin(top_edit)]

if px:
    fig = px.line(edit_top, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(edit_top.head(50))

## 14. Sep 2026 Forecast Detail

Forecast ini memakai grain `Customer x SKU`, dengan baseline yang dibersihkan dari suspected promo/outlier secara konservatif.

Parameter utama:
- `FORECAST_MONTH`: bulan yang akan diforecast
- `RUNRATE_MONTH`: bulan berjalan yang masih MTD
- `CF Edit`: volume yang sudah disetarakan SKU dan pack

In [ ]:
FORECAST_MONTH = pd.Timestamp("2026-09-01")
RUNRATE_MONTH = FORECAST_MONTH - pd.DateOffset(months=1)

key_cols = [
    "Channel Group", "Branch", "Cust Code", "Customer Name",
    "Item Code Edit", "Short Item Description Edit", "SKU Edit",
]

daily_customer_sku = (
    sales_aligned_append
    .groupby(["Date", "Month", *key_cols], as_index=False)
    .agg(CF=("CF Edit", "sum"))
)

max_data_date = daily_customer_sku["Date"].max()
if max_data_date.to_period("M").to_timestamp() == RUNRATE_MONTH:
    elapsed_days = max_data_date.day
else:
    elapsed_days = daily_customer_sku.loc[daily_customer_sku["Month"].eq(RUNRATE_MONTH), "Date"].dt.day.max()
    elapsed_days = 0 if pd.isna(elapsed_days) else int(elapsed_days)

total_days = RUNRATE_MONTH.days_in_month
runrate_multiplier = total_days / elapsed_days if elapsed_days else 0

monthly_customer_sku = (
    daily_customer_sku
    .groupby(["Month", *key_cols], as_index=False)
    .agg(CF=("CF", "sum"))
)

print("Forecast month:", FORECAST_MONTH.date())
print("Run-rate month:", RUNRATE_MONTH.date())
print("Max data date:", max_data_date.date())
print("Run-rate elapsed days:", elapsed_days)
print("Run-rate multiplier:", round(runrate_multiplier, 4))
display(monthly_customer_sku.head())

## 15. Conservative Promo/Outlier Cleaning

In [ ]:
forecast_input_months = pd.date_range("2026-03-01", RUNRATE_MONTH, freq="MS")
history_months = pd.date_range("2026-01-01", RUNRATE_MONTH, freq="MS")

wide = monthly_customer_sku.pivot_table(
    index=key_cols,
    columns="Month",
    values="CF",
    aggfunc="sum",
    fill_value=0,
).reset_index()

for month in history_months:
    if month not in wide.columns:
        wide[month] = 0

wide = wide[key_cols + list(history_months)]

def clean_months(row):
    values = {m: float(row[m]) for m in history_months}
    clean = {}
    promo_flags = {}
    promo_uplift = {}

    for i, month in enumerate(history_months):
        actual = values[month]
        previous = [clean[m] for m in history_months[max(0, i - 3):i] if clean.get(m, 0) > 0]
        if len(previous) >= 2:
            med = float(np.median(previous))
            std = float(np.std(previous))
            threshold = max(med * 2.0, med + 2.0 * std)
            is_promo = med > 0 and actual >= 10 and actual > threshold
            clean_value = min(actual, med) if is_promo else actual
        else:
            is_promo = False
            clean_value = actual
        clean[month] = clean_value
        promo_flags[month] = is_promo
        promo_uplift[month] = max(0, actual - clean_value)

    aug_actual = values.get(RUNRATE_MONTH, 0)
    aug_clean_mtd = clean.get(RUNRATE_MONTH, 0)
    aug_rr = aug_actual * runrate_multiplier
    aug_clean_rr = aug_clean_mtd * runrate_multiplier

    return pd.Series({
        **{f"{m.strftime('%b')}_Actual": values[m] for m in forecast_input_months},
        **{f"{m.strftime('%b')}_Clean": (aug_clean_rr if m == RUNRATE_MONTH else clean[m]) for m in forecast_input_months},
        "Aug_MTD": aug_actual,
        "Aug_RunRate": aug_rr,
        "Aug_Clean_RunRate": aug_clean_rr,
        "Promo_Suspect_Months": int(sum(promo_flags.values())),
        "Promo_Suspect_Uplift": float(sum(promo_uplift.values())),
    })

clean_features = wide.apply(clean_months, axis=1)
forecast_base = pd.concat([wide[key_cols].reset_index(drop=True), clean_features], axis=1)

display(forecast_base.head())

## 16. Segmented Robust Forecast

In [ ]:
recent_cols = ["Mar_Clean", "Apr_Clean", "May_Clean", "Jun_Clean", "Jul_Clean", "Aug_Clean"]
recent_3_cols = ["Jun_Clean", "Jul_Clean", "Aug_Clean"]

def nonzero(values):
    return [float(v) for v in values if float(v) > 0]

def trimmed_mean(values):
    vals = [float(v) for v in values]
    if len(vals) >= 4:
        vals = sorted(vals)[1:-1]
    return float(np.mean(vals)) if vals else 0.0

def classify_and_forecast(row):
    recent = [row[c] for c in recent_cols]
    recent_3 = [row[c] for c in recent_3_cols]
    nz = nonzero(recent)
    active_months = len(nz)
    total_recent = float(sum(recent))
    avg = float(np.mean(nz)) if nz else 0.0
    volatility = float(np.std(nz) / avg) if avg > 0 and len(nz) >= 2 else 0.0

    median_3m = float(np.median(nonzero(recent_3))) if nonzero(recent_3) else 0.0
    median_6m = float(np.median(nz)) if nz else 0.0
    weighted_recent = float(0.20 * row["Jun_Clean"] + 0.30 * row["Jul_Clean"] + 0.50 * row["Aug_Clean"])
    trimmed_6m = trimmed_mean(recent)

    if total_recent == 0:
        demand_class = "Inactive"
        raw_fc = 0.0
    elif active_months <= 2:
        demand_class = "New / Sparse"
        raw_fc = max(median_6m, row["Aug_Clean"] * 0.75)
    elif active_months <= 3 or volatility > 1.50:
        demand_class = "Intermittent"
        raw_fc = median_6m
    elif volatility > 0.75:
        demand_class = "Volatile"
        raw_fc = 0.50 * median_3m + 0.50 * trimmed_6m
    else:
        demand_class = "Stable"
        raw_fc = 0.60 * weighted_recent + 0.40 * median_6m

    max_recent = max(recent) if recent else 0.0
    upper_guardrail = max_recent * 1.25 if max_recent > 0 else raw_fc
    base_fc = min(max(raw_fc, 0.0), upper_guardrail)
    low_fc = base_fc * 0.90
    high_fc = base_fc * 1.15

    return pd.Series({
        "Active Months": active_months,
        "Volatility": volatility,
        "Demand Class": demand_class,
        "Median 3M": median_3m,
        "Median 6M": median_6m,
        "Weighted Recent": weighted_recent,
        "Trimmed Mean 6M": trimmed_6m,
        "Raw Forecast": raw_fc,
        "Guardrail Cap": upper_guardrail,
        "Sep Forecast Base": base_fc,
        "Sep Forecast Low": low_fc,
        "Sep Forecast High": high_fc,
    })

forecast_metrics = forecast_base.apply(classify_and_forecast, axis=1)
forecast_detail_sep = pd.concat([forecast_base, forecast_metrics], axis=1)
forecast_detail_sep = forecast_detail_sep.sort_values("Sep Forecast Base", ascending=False)

display(forecast_detail_sep.head(30))

## 17. Sep Forecast Summary

In [ ]:
summary_sep = pd.DataFrame({
    "Metric": [
        "Rows",
        "Sep Forecast Base",
        "Sep Forecast Low",
        "Sep Forecast High",
        "Promo Suspect Months",
        "Promo Suspect Uplift",
    ],
    "Value": [
        len(forecast_detail_sep),
        forecast_detail_sep["Sep Forecast Base"].sum(),
        forecast_detail_sep["Sep Forecast Low"].sum(),
        forecast_detail_sep["Sep Forecast High"].sum(),
        forecast_detail_sep["Promo_Suspect_Months"].sum(),
        forecast_detail_sep["Promo_Suspect_Uplift"].sum(),
    ],
})

summary_by_class_sep = (
    forecast_detail_sep
    .groupby("Demand Class", as_index=False)
    .agg(
        rows=("SKU Edit", "size"),
        sep_base=("Sep Forecast Base", "sum"),
        sep_low=("Sep Forecast Low", "sum"),
        sep_high=("Sep Forecast High", "sum"),
        promo_months=("Promo_Suspect_Months", "sum"),
    )
    .sort_values("sep_base", ascending=False)
)

summary_by_sku_sep = (
    forecast_detail_sep
    .groupby(["Item Code Edit", "Short Item Description Edit", "SKU Edit"], as_index=False)
    .agg(
        rows=("Cust Code", "size"),
        sep_base=("Sep Forecast Base", "sum"),
        sep_low=("Sep Forecast Low", "sum"),
        sep_high=("Sep Forecast High", "sum"),
    )
    .sort_values("sep_base", ascending=False)
)

display(summary_sep)
display(summary_by_class_sep)
display(summary_by_sku_sep.head(30))

if px:
    fig = px.bar(summary_by_sku_sep.head(20), x="SKU Edit", y="sep_base", title="Sep 2026 Forecast Base by SKU - Top 20")
    fig.show()

## 18. Manual Filter

In [ ]:
keyword = "VOLT"

filtered = sales_aligned_append[
    sales_aligned_append["SKU Real"].str.contains(keyword, case=False, na=False)
    | sales_aligned_append["SKU Edit"].str.contains(keyword, case=False, na=False)
].copy()

filtered_long = pd.concat(
    [
        filtered.assign(SKU_View="Real", SKU=filtered["SKU Real"], CF_View=filtered["CF Real"]),
        filtered.assign(SKU_View="Edit", SKU=filtered["SKU Edit"], CF_View=filtered["CF Edit"]),
    ],
    ignore_index=True,
)

filtered_monthly = filtered_long.groupby(["Month", "SKU_View", "SKU"], as_index=False).agg(CF=("CF_View", "sum"))
display(filtered_monthly)

if px:
    fig = px.line(
        filtered_monthly,
        x="Month",
        y="CF",
        color="SKU",
        line_dash="SKU_View",
        markers=True,
        title=f"Manual Filter: {keyword} - Real vs Edit",
    )
    fig.show()